# Study Telemetry — Exploratory Data Analysis

Loads the Google Sheets export and runs the standard checks for the
SpeechSpectrum Prolific study:

1. **Recruitment & dropout** — participants who reach each stage of the funnel.
2. **Condition balance** — random assignment to the 5 between-subjects conditions.
3. **Latin-square balance** — random assignment to the 4 conversation orders.
4. **Timing** — per-page, per-transcript, per-question.
5. **Realism ratings & comprehension check.**
6. **Demographics.**
7. **Comprehension accuracy** — per condition × transcript (the primary outcome).

## How to use

1. In the Google Sheet that receives the webhook, **File → Download → CSV (.csv)**.
2. Save it as `study_telemetry.csv` in this `notebooks/` directory.
3. Run all cells.

Each row of the sheet should look like:

| column | example |
|---|---|
| `timestamp` | `2026-05-08T19:34:11.123Z` |
| `event_type` | `condition_assignment` / `irb_consent` / `demographic` / `content_ack` / `study_start` / `question_timing` / `question_order` / `survey_response` / `transcript_timing` / `style_switch` / `style_duration` / `practice_response` / `post_survey` / `study_complete` / `study_exit` / `bug_report` / `page_timing` |
| `prolific_id` | `TESTabc` or real ID |
| `assigned_condition` | `enhanced / A,B,C,D` |
| `latin_square_order` | `A` / `B` / `C` / `D` |
| `question_id` | depends on event |
| `question_text` | depends on event |
| `response` | depends on event |
| `extra` | JSON or freeform |

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 80)

CSV_PATH = Path('study_telemetry.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'Place the Sheet CSV export at {CSV_PATH.resolve()}.\n'
        'In the linked Google Sheet: File → Download → CSV.'
    )

df = pd.read_csv(CSV_PATH)
# Normalize columns (in case the sheet has missing/extra columns)
for col in ['timestamp','event_type','prolific_id','assigned_condition',
            'latin_square_order','question_id','question_text','response','extra']:
    if col not in df.columns:
        df[col] = ''
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
# Drop TEST participants by default (toggle as needed)
EXCLUDE_TEST = True
if EXCLUDE_TEST:
    df = df[~df['prolific_id'].fillna('').str.upper().str.startswith('TEST')].copy()
# Split style and order code from assigned_condition ("style / A,B,C,D")
df['style'] = df['assigned_condition'].fillna('').str.split(' / ').str[0]
print(f'{len(df)} rows; {df["prolific_id"].nunique()} unique participants')
df['event_type'].value_counts()

## 1. Recruitment & dropout funnel

How many participants reached each stage?

In [ ]:
stage_events = {
    'consent_seen':      ('page_timing', 'consent'),       # logged on consent click-through
    'consent_agreed':    ('irb_consent', None),
    'pid_entered':       ('condition_assignment', None),
    'demographics_done': ('page_timing', 'demographics'),
    'content_ack_done':  ('page_timing', 'content_ack'),
    'study_start':       ('study_start', None),
    'study_complete':    ('study_complete', None),
    'post_survey_done':  ('post_survey', None),
}

def participants_at(stage):
    evt, qid = stage_events[stage]
    sub = df[df['event_type'] == evt]
    if qid is not None:
        sub = sub[sub['question_id'] == qid]
    return set(sub['prolific_id'].dropna()) - {''}

funnel = {s: len(participants_at(s)) for s in stage_events}
funnel_df = pd.DataFrame(list(funnel.items()), columns=['stage','n'])
display(funnel_df)

fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(funnel_df, x='n', y='stage', ax=ax, color='#4F46E5')
for i, v in enumerate(funnel_df['n']):
    ax.text(v + 0.3, i, str(v), va='center')
ax.set_title('Recruitment funnel')
ax.set_xlabel('participants reaching stage')
plt.tight_layout(); plt.show()

## 2. Condition balance (5 between-subjects conditions)

Use the `condition_assignment` event with `first_assignment=true` to count unique participants per condition.

In [ ]:
def first_assignment_rows(df):
    sub = df[df['event_type'] == 'condition_assignment'].copy()
    def is_first(extra):
        try: return json.loads(extra).get('first_assignment', True)
        except Exception: return True
    sub['first'] = sub['extra'].fillna('').apply(is_first)
    # If a participant has any first=True row, keep the earliest
    sub = sub.sort_values('timestamp')
    return sub[sub['first']].drop_duplicates('prolific_id', keep='first')

assign = first_assignment_rows(df)

fig, ax = plt.subplots(figsize=(7,4))
order_styles = ['verbatim','non-verbatim','enhanced','bullet-points','demo']
counts = assign['style'].value_counts().reindex(order_styles, fill_value=0)
sns.barplot(x=counts.index, y=counts.values, ax=ax, color='#4F46E5')
for i, v in enumerate(counts.values):
    ax.text(i, v + 0.2, str(v), ha='center')
ax.set_title('Participants per condition (between-subjects)')
ax.set_xlabel(''); ax.set_ylabel('n')
plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

## 3. Latin-square order balance

Each condition should be roughly evenly split across the 4 orders (target = 9 per cell at full recruitment).

In [ ]:
def extract_order_key(extra):
    try: return json.loads(extra).get('latin_square_order')
    except Exception: return None

assign['order_key'] = assign['extra'].fillna('').apply(extract_order_key)
cross = pd.crosstab(assign['style'], assign['order_key']).reindex(
    index=order_styles, columns=['A','B','C','D'], fill_value=0)
display(cross)

fig, ax = plt.subplots(figsize=(6,4))
sns.heatmap(cross, annot=True, fmt='d', cmap='Purples', cbar=False, ax=ax)
ax.set_title('Latin-square assignment (condition × order)')
ax.set_xlabel('order'); ax.set_ylabel('condition')
plt.tight_layout(); plt.show()

## 4. Timing distributions

In [ ]:
# Page timing (consent / pid / demographics / content_ack)
page_timing = df[df['event_type'] == 'page_timing'].copy()
page_timing['seconds'] = pd.to_numeric(page_timing['response'], errors='coerce')

fig, ax = plt.subplots(figsize=(8,4))
sns.boxplot(page_timing, x='question_id', y='seconds',
            order=['consent','pid','demographics','content_ack'], ax=ax, color='#A5B4FC')
ax.set_yscale('log')
ax.set_title('Per-page time spent (log scale)')
ax.set_xlabel('page'); ax.set_ylabel('seconds')
plt.tight_layout(); plt.show()

In [ ]:
# Per-transcript timing
tt = df[df['event_type'] == 'transcript_timing'].copy()
def transcript_seconds(extra):
    try: return json.loads(extra).get('duration_seconds') or json.loads(extra).get('seconds')
    except Exception: return None
tt['seconds'] = pd.to_numeric(tt['response'], errors='coerce')
if tt['seconds'].isna().all():
    tt['seconds'] = tt['extra'].fillna('').apply(transcript_seconds)

fig, ax = plt.subplots(figsize=(8,4))
if not tt.empty:
    sns.boxplot(tt, x='style', y='seconds', order=order_styles, ax=ax, color='#A5B4FC')
    ax.set_title('Per-transcript time, by condition')
    ax.set_ylabel('seconds')
    plt.xticks(rotation=15)
plt.tight_layout(); plt.show()

In [ ]:
# Per-question timing
qt = df[df['event_type'] == 'question_timing'].copy()
def question_seconds(extra):
    try:
        d = json.loads(extra)
        return d.get('duration_seconds') or d.get('seconds')
    except Exception:
        return None
qt['seconds'] = qt['extra'].fillna('').apply(question_seconds)
qt['seconds'] = pd.to_numeric(qt['seconds'], errors='coerce')

if not qt.empty:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(qt['seconds'].dropna(), bins=60, ax=ax, color='#4F46E5')
    ax.set_xlim(0, qt['seconds'].quantile(0.99))
    ax.set_title('Per-question time (clipped at 99th pct)')
    ax.set_xlabel('seconds')
    plt.tight_layout(); plt.show()

## 5. Realism ratings & comprehension check

In [ ]:
ps = df[df['event_type'] == 'post_survey'].copy()
realism = ps[ps['question_id'].str.startswith('ps_realism_Transcr', na=False)].copy()
realism['transcript'] = realism['question_id'].str.replace('ps_realism_','', regex=False)
realism['rating'] = pd.to_numeric(realism['response'], errors='coerce')

if not realism.empty:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.boxplot(realism, x='transcript', y='rating',
                order=['TranscrA','TranscrB','TranscrC','TranscrD'], ax=ax, color='#A5B4FC')
    ax.set_title('Realism rating per hearing (1=not real, 5=authentic)')
    ax.set_ylim(0.5, 5.5)
    plt.tight_layout(); plt.show()

# Comprehension-check open responses
comp = ps[ps['question_id'] == 'ps_last_transcript'].copy()
print(f'{len(comp)} comprehension-check responses (last transcript description)')
if not comp.empty:
    display(comp[['prolific_id','response']].head(10))

## 6. Demographics

In [ ]:
demo = df[df['event_type'] == 'demographic'].copy()
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (qid, sub) in zip(axes.flat, demo.groupby('question_id')):
    counts = sub['response'].value_counts()
    counts.plot.barh(ax=ax, color='#4F46E5')
    ax.set_title(qid, fontsize=10)
    ax.set_xlabel('')
# Hide unused subplots
for ax in axes.flat[len(demo['question_id'].unique()):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Comprehension accuracy — the primary outcome

Join the `survey_response` rows against the question key in `docs/questions.js` and score each item. Then aggregate to condition × transcript for the 5×4 mixed ANOVA.

In [ ]:
import re
QJS = Path('../docs/questions.js').read_text(encoding='utf-8')
m = re.search(r'QUESTIONS_BY_TRANSCRIPT\s*=\s*(\{.*?\});', QJS, re.S)
if not m:
    raise RuntimeError('Could not parse QUESTIONS_BY_TRANSCRIPT.')
QBT = json.loads(m.group(1))
key = {}  # question_id -> (transcript, correct_option_text)
for tkey, qs in QBT.items():
    for q in qs:
        ci = q.get('correct_index')
        if ci is None: continue
        key[q['id']] = (tkey, q['options'][ci])
print(f'{len(key)} keyed questions across {len(QBT)} transcripts')

resp = df[df['event_type'] == 'survey_response'].copy()
def score(row):
    qid = row['question_id']
    if qid not in key: return None
    _, correct = key[qid]
    return int(str(row['response']).strip() == str(correct).strip())
resp['correct'] = resp.apply(score, axis=1)
resp['transcript'] = resp['question_id'].str.split('_').str[0]

acc = (resp.dropna(subset=['correct'])
        .groupby(['style','transcript'])['correct']
        .agg(['mean','count']).reset_index())
display(acc)

if not acc.empty:
    fig, ax = plt.subplots(figsize=(8,5))
    pivot = acc.pivot(index='transcript', columns='style', values='mean')
    pivot = pivot.reindex(index=['TranscrA','TranscrB','TranscrC','TranscrD'],
                          columns=order_styles)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1, ax=ax)
    ax.set_title('Mean comprehension accuracy (condition × transcript)')
    plt.tight_layout(); plt.show()

## Notes

- `EXCLUDE_TEST = True` in the first cell drops PIDs starting with `TEST`. Flip to `False` to inspect internal testing rows.
- The comprehension scorer in Section 7 matches the participant's `response` string against the `correct_index`-indexed option text in `docs/questions.js`. If you re-run `scripts/build_questions.py` and re-deploy, scoring will continue to work as long as option text doesn't change retroactively for already-collected responses.
- For the formal 5×4 mixed ANOVA on accuracy, use `statsmodels` or `pingouin` on the participant-level long-format frame (`prolific_id`, `style`, `transcript`, `accuracy`).